In [9]:
import numpy as np
import pandas as pd

In [10]:
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "Comment_Generation"


print("Project root:", PROJECT_ROOT)
print("Data folder:", DATA_DIR)
print("Data folder exists?", DATA_DIR.exists())
print()
print("Files in data folder:")
for f in DATA_DIR.iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:20s}  {size_mb:.1f} MB")

Project root: d:\College\NCI\Semester 2\Practicum\Semester 3\code\cot-code-review
Data folder: d:\College\NCI\Semester 2\Practicum\Semester 3\code\cot-code-review\data\raw\Comment_Generation
Data folder exists? True

Files in data folder:
  msg-test.jsonl        262.5 MB
  msg-train.jsonl       3453.9 MB
  msg-valid.jsonl       261.1 MB


In [11]:
import json

# Test file path
test_file = DATA_DIR / "msg-test.jsonl"


print("First 5 samples from msg-test.jsonl")

with open(test_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        sample = json.loads(line)
        print(f"\n--- Sample {i+1} ---")
        print(f"Keys available: {list(sample.keys())}")
        print(f"Sample preview:")
        for key, value in sample.items():
            value_str = str(value)
            if len(value_str) > 150:
                value_str = value_str[:150] + "..."
            print(f"  {key}: {value_str}")

First 5 samples from msg-test.jsonl

--- Sample 1 ---
Keys available: ['patch', 'y', 'oldf', 'idx', 'id', 'msg', 'proj', 'lang']
Sample preview:
  patch: @@ -53,7 +53,7 @@ public class ProtocGapicPluginGeneratorTest {
                 model.getFiles().stream().map(ProtoFile::getProto).collect(Collectors...
  y: 1
  oldf: /* Copyright 2019 Google LLC
 *
 * Licensed under the Apache License, Version 2.0 (the "License");
 * you may not use this file except in compliance w...
  idx: 1
  id: 30879
  msg: can we also test for `transport=rest`?
  proj: googleapis-gapic-generator
  lang: java

--- Sample 2 ---
Keys available: ['patch', 'y', 'oldf', 'idx', 'id', 'msg', 'proj', 'lang']
Sample preview:
  patch: @@ -182,7 +182,9 @@ abstract class AbstractSolrBackendFactory implements FactoryInterface
      */
     protected function createBackend(Connector $co...
  y: 1
  oldf: <?php

/**
 * Abstract factory for SOLR backends.
 *
 * PHP version 7
 *
 * Copyright (C) Villanova University 2013.
 *
 

In [12]:
# Count samples by language
from collections import Counter

test_file = DATA_DIR / "msg-test.jsonl"

lang_counter = Counter()
total = 0

print("Counting languages in msg-test.jsonl")
with open(test_file, 'r', encoding='utf-8') as f:
    for line in f:
        sample = json.loads(line)
        lang_counter[sample['lang']] += 1
        total += 1

print(f"\nTotal samples in test: {total:,}")
print(f"\nLanguage distribution:")
print("-" * 40)
for lang, count in lang_counter.most_common():
    percentage = (count / total) * 100
    print(f"  {lang:10s} : {count:>6,} samples  ({percentage:.1f}%)")

Counting languages in msg-test.jsonl

Total samples in test: 10,169

Language distribution:
----------------------------------------
  go         :  2,840 samples  (27.9%)
  java       :  1,658 samples  (16.3%)
  py         :  1,403 samples  (13.8%)
  rb         :  1,000 samples  (9.8%)
  js         :    967 samples  (9.5%)
  cpp        :    731 samples  (7.2%)
  .cs        :    691 samples  (6.8%)
  c          :    453 samples  (4.5%)
  php        :    426 samples  (4.2%)


In [13]:
# Robust language counter, handles missing 'lang' field

def count_languages(filepath, name):
    """Count languages in JSONL file, handling missing 'lang' field."""
    print(f"\nCounting languages in {name}...")
    
    lang_counter = Counter()
    total = 0
    missing_lang = 0  # Track samples without 'lang' field
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            sample = json.loads(line)
            
            # Defensive: check if 'lang' field exists
            lang = sample.get('lang', None)  # Returns None if missing
            
            if lang is None:
                missing_lang += 1
            else:
                lang_counter[lang] += 1
            
            total += 1
            
            # Progress indicator every 50K samples
            if total % 50000 == 0:
                print(f"  Processed {total:,} samples")
    
    print(f"\n  Total samples: {total:,}")
    print(f"  Samples with 'lang' field: {total - missing_lang:,}")
    print(f"  Samples MISSING 'lang' field: {missing_lang:,}")
    print(f"  Python ('py') samples: {lang_counter.get('py', 0):,}")
    return lang_counter, total, missing_lang


# Run for both files
train_langs, train_total, train_missing = count_languages(train_file, "msg-train.jsonl")
valid_langs, valid_total, valid_missing = count_languages(valid_file, "msg-valid.jsonl")

# Summary table
print("SUMMARY, Python Samples Across All Files")
print(f"{'File':<20} {'Total':>10} {'Python':>10} {'Missing lang':>15}")



test_py = 1403
test_total = 10169
test_missing = 0

print(f"{'msg-test.jsonl':<20} {test_total:>10,} {test_py:>10,} {test_missing:>15,}")
print(f"{'msg-valid.jsonl':<20} {valid_total:>10,} {valid_langs.get('py', 0):>10,} {valid_missing:>15,}")
print(f"{'msg-train.jsonl':<20} {train_total:>10,} {train_langs.get('py', 0):>10,} {train_missing:>15,}")

grand_total_py = test_py + valid_langs.get('py', 0) + train_langs.get('py', 0)
grand_total = test_total + valid_total + train_total
print(f"{'TOTAL':<20} {grand_total:>10,} {grand_total_py:>10,} {train_missing + valid_missing:>15,}")


Counting languages in msg-train.jsonl...
  Processed 50,000 samples
  Processed 100,000 samples

  Total samples: 117,739
  Samples with 'lang' field: 0
  Samples MISSING 'lang' field: 117,739
  Python ('py') samples: 0

Counting languages in msg-valid.jsonl...

  Total samples: 10,319
  Samples with 'lang' field: 10,319
  Samples MISSING 'lang' field: 0
  Python ('py') samples: 1,420
SUMMARY, Python Samples Across All Files
File                      Total     Python    Missing lang
msg-test.jsonl           10,169      1,403               0
msg-valid.jsonl          10,319      1,420               0
msg-train.jsonl         117,739          0         117,739
TOTAL                   138,227      2,823         117,739


In [15]:
# Train file mein koi bhi sample lang ya proj rakhta hai kya — full scan
lang_present = 0
proj_present = 0
all_keys = set()

with open(train_file, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        all_keys.update(s.keys())
        if 'lang' in s: lang_present += 1
        if 'proj' in s: proj_present += 1

print(f"Train samples with 'lang' : {lang_present:,}")
print(f"Train samples with 'proj' : {proj_present:,}")
print(f"\nTrain mein jitne bhi unique fields mile: {sorted(all_keys)}")

Train samples with 'lang' : 0
Train samples with 'proj' : 0

Train mein jitne bhi unique fields mile: ['id', 'msg', 'oldf', 'patch', 'y']


In [16]:
!pip install remotezip -q

from remotezip import RemoteZip

urls = {
    "Code_Refinement":        "https://zenodo.org/records/6900648/files/Code_Refinement.zip?download=1",
    "Diff_Quality_Estimation":"https://zenodo.org/records/6900648/files/Diff_Quality_Estimation.zip?download=1",
}

zip_contents = {}
for name, url in urls.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    try:
        with RemoteZip(url) as z:
            names = z.namelist()
            zip_contents[name] = (url, names)
            for n in names:
                print("  ", n)
    except Exception as e:
        print("  ERROR:", e)

Code_Refinement
   Code_Refinement/
   Code_Refinement/ref-test.jsonl
   Code_Refinement/ref-train.jsonl
   Code_Refinement/ref-valid.jsonl
Diff_Quality_Estimation
   Diff_Quality_Estimation/
   Diff_Quality_Estimation/cls-test.jsonl
   Diff_Quality_Estimation/cls-train-chunk-0.jsonl
   Diff_Quality_Estimation/cls-train-chunk-2.jsonl
   Diff_Quality_Estimation/cls-valid.jsonl
   Diff_Quality_Estimation/cls-train-chunk-1.jsonl
   Diff_Quality_Estimation/cls-train-chunk-3.jsonl


In [17]:
import json

def peek_train_file(url, member_name):
    """Remote zip ke andar ek file ki pehli line padho — bina poora download kiye."""
    with RemoteZip(url) as z:
        with z.open(member_name) as f:
            first_line = f.readline()
    sample = json.loads(first_line)
    return sample

# Step 1 ke output se train file ka exact naam yahan daal
# (jo bhi 'train' wala naam dikhe usko use kar)
for name, (url, names) in zip_contents.items():
    train_members = [n for n in names if 'train' in n.lower() and n.endswith('.jsonl')]
    for member in train_members:
        print("=" * 60)
        print(f"{name}  →  {member}")
        print("=" * 60)
        try:
            sample = peek_train_file(url, member)
            keys = list(sample.keys())
            print("Fields:", keys)
            print("  lang present? ", 'lang' in sample)
            print("  proj present? ", 'proj' in sample)
            print("  id present?   ", 'id' in sample)
        except Exception as e:
            print("  ERROR:", e)

Code_Refinement  →  Code_Refinement/ref-train.jsonl
Fields: ['old_hunk', 'oldf', 'hunk', 'comment', 'ids', 'repo', 'ghid', 'old', 'new', 'lang']
  lang present?  True
  proj present?  False
  id present?    False
Diff_Quality_Estimation  →  Diff_Quality_Estimation/cls-train-chunk-0.jsonl
Fields: ['oldf', 'patch', 'msg', 'id', 'y']
  lang present?  False
  proj present?  False
  id present?    True
Diff_Quality_Estimation  →  Diff_Quality_Estimation/cls-train-chunk-2.jsonl
Fields: ['oldf', 'patch', 'msg', 'id', 'y']
  lang present?  False
  proj present?  False
  id present?    True
Diff_Quality_Estimation  →  Diff_Quality_Estimation/cls-train-chunk-1.jsonl
Fields: ['patch', 'y', 'oldf', 'idx', 'id', 'msg', 'proj', 'lang']
  lang present?  True
  proj present?  True
  id present?    True
Diff_Quality_Estimation  →  Diff_Quality_Estimation/cls-train-chunk-3.jsonl
Fields: ['oldf', 'patch', 'msg', 'id', 'y']
  lang present?  False
  proj present?  False
  id present?    True


In [18]:
# msg-train ke ids ka set banao
msg_train_ids = set()
with open(train_file, 'r', encoding='utf-8') as f:
    for line in f:
        msg_train_ids.add(json.loads(line)['id'])

print(f"msg-train mein total ids: {len(msg_train_ids):,}")

msg-train mein total ids: 76,885


In [19]:
import json
from remotezip import RemoteZip

dqe_url = "https://zenodo.org/records/6900648/files/Diff_Quality_Estimation.zip?download=1"
member  = "Diff_Quality_Estimation/cls-train-chunk-1.jsonl"

# chunk-1 se id -> (lang, proj) map banao (sirf is member ke bytes download honge)
id_to_meta = {}
print("Streaming cls-train-chunk-1 ...")
with RemoteZip(dqe_url) as z:
    with z.open(member) as f:
        for i, line in enumerate(f):
            s = json.loads(line)
            if 'id' in s and 'lang' in s:
                id_to_meta[s['id']] = (s.get('lang'), s.get('proj'))
            if (i + 1) % 50000 == 0:
                print(f"  {i+1:,} lines read, {len(id_to_meta):,} mapped")

print(f"\nchunk-1 mein unique ids with lang/proj : {len(id_to_meta):,}")

Streaming cls-train-chunk-1 ...
  50,000 lines read, 16,449 mapped

chunk-1 mein unique ids with lang/proj : 19,873


In [20]:
# Overlap check
overlap = msg_train_ids & set(id_to_meta.keys())
print(f"msg-train unique ids   : {len(msg_train_ids):,}")
print(f"chunk-1 mapped ids     : {len(id_to_meta):,}")
print(f"OVERLAP                : {len(overlap):,}  ({len(overlap)/len(msg_train_ids)*100:.1f}%)")

msg-train unique ids   : 76,885
chunk-1 mapped ids     : 19,873
OVERLAP                : 11,817  (15.4%)


In [21]:
from collections import Counter

matched_langs = Counter(id_to_meta[i][0] for i in overlap)
total_matched = sum(matched_langs.values())

print("Matched ids ka language distribution:")
print("-" * 40)
for lang, cnt in matched_langs.most_common():
    print(f"  {lang:10s} : {cnt:>6,}  ({cnt/total_matched*100:.1f}%)")

py_rate = matched_langs.get('py', 0) / total_matched * 100
print(f"\nPython rate: {py_rate:.1f}%")
print("Test file Python rate tha: 13.8%")
print("→ Agar ye ~10-18% range mein hai, join REAL hai ✅")
print("→ Agar random/weird distribution hai, id sirf row-index hai, join FAKE ❌")

Matched ids ka language distribution:
----------------------------------------
  py         :  2,677  (22.7%)
  go         :  2,377  (20.1%)
  java       :  2,223  (18.8%)
  js         :  1,460  (12.4%)
  cpp        :    986  (8.3%)
  .cs        :    896  (7.6%)
  php        :    520  (4.4%)
  rb         :    502  (4.2%)
  c          :    176  (1.5%)

Python rate: 22.7%
Test file Python rate tha: 13.8%
→ Agar ye ~10-18% range mein hai, join REAL hai ✅
→ Agar random/weird distribution hai, id sirf row-index hai, join FAKE ❌


In [22]:
import json
from remotezip import RemoteZip

ref_url = "https://zenodo.org/records/6900648/files/Code_Refinement.zip?download=1"
member  = "Code_Refinement/ref-train.jsonl"

# Pehle 3 samples ke key-fields inspect karo
print("ref-train ke pehle 3 samples ke join-candidate fields:\n")
with RemoteZip(ref_url) as z:
    with z.open(member) as f:
        for i, line in enumerate(f):
            if i >= 3:
                break
            s = json.loads(line)
            print(f"--- Sample {i} ---")
            print(f"  ids  : {s.get('ids')}   (type: {type(s.get('ids')).__name__})")
            print(f"  ghid : {s.get('ghid')}  (type: {type(s.get('ghid')).__name__})")
            print(f"  repo : {s.get('repo')}")
            print(f"  lang : {s.get('lang')}")
            print()

# Comparison ke liye msg-train ka ek id yaad dila
sample_msg_id = next(iter(msg_train_ids))
print(f"Reference — msg-train ka ek id: {sample_msg_id}  (type: {type(sample_msg_id).__name__})")

ref-train ke pehle 3 samples ke join-candidate fields:

--- Sample 0 ---
  ids  : [24794, '17d785e3dce428351b39d1290ec28a383342c8cb', '91569fc31a46c739ae5ccf18437d5b4782ca3c3a']   (type: list)
  ghid : 3277  (type: int)
  repo : KhronosGroup/SPIRV-Tools
  lang : cpp

--- Sample 1 ---
  ids  : [20887, '3e39aa16bb601c90e73b98249a9dcd77f17d511e', '9ee2d9cfbcafbce2b81669cc6adf3936e4b7b94f']   (type: list)
  ghid : 1602  (type: int)
  repo : wikimedia/apps-android-wikipedia
  lang : java

--- Sample 2 ---
  ids  : [81755, '1f6f246f4a1ccc855ecefe34e316d8bbb83512c0', '2948445f139ceb4b435465d3715a73c6f105acc7']   (type: list)
  ghid : 9972  (type: int)
  repo : discourse/discourse
  lang : rb

Reference — msg-train ka ek id: 262144  (type: int)


In [23]:
from collections import Counter

ref_ghid_to_meta = {}   # ghid -> (lang, repo)
ref_ids_to_meta  = {}   # individual id (ids list se) -> (lang, repo)

print("Streaming ref-train ...")
with RemoteZip(ref_url) as z:
    with z.open(member) as f:
        for i, line in enumerate(f):
            s = json.loads(line)
            lang, repo = s.get('lang'), s.get('repo')

            # ghid based
            if 'ghid' in s and s['ghid'] is not None:
                ref_ghid_to_meta[s['ghid']] = (lang, repo)

            # ids based (agar list hai toh expand karo)
            ids_val = s.get('ids')
            if isinstance(ids_val, list):
                for single_id in ids_val:
                    ref_ids_to_meta[single_id] = (lang, repo)
            elif ids_val is not None:
                ref_ids_to_meta[ids_val] = (lang, repo)

            if (i + 1) % 30000 == 0:
                print(f"  {i+1:,} lines read")

print(f"\nref unique ghids : {len(ref_ghid_to_meta):,}")
print(f"ref unique ids   : {len(ref_ids_to_meta):,}")

# Dono keys pe overlap test
ov_ghid = msg_train_ids & set(ref_ghid_to_meta.keys())
ov_ids  = msg_train_ids & set(ref_ids_to_meta.keys())

print(f"\nmsg-train ids                : {len(msg_train_ids):,}")
print(f"OVERLAP via ghid             : {len(ov_ghid):,}  ({len(ov_ghid)/len(msg_train_ids)*100:.1f}%)")
print(f"OVERLAP via ids              : {len(ov_ids):,}  ({len(ov_ids)/len(msg_train_ids)*100:.1f}%)")

Streaming ref-train ...
  30,000 lines read
  60,000 lines read
  90,000 lines read
  120,000 lines read
  150,000 lines read

ref unique ghids : 25,930
ref unique ids   : 339,480

msg-train ids                : 76,885
OVERLAP via ghid             : 9,653  (12.6%)
OVERLAP via ids              : 42,013  (54.6%)


In [24]:
# Jo bhi overlap zyada/valid nikla, usko 'winner' map maano. Example ghid:
winner_map = ref_ghid_to_meta if len(ov_ghid) >= len(ov_ids) else ref_ids_to_meta
winner_ov  = ov_ghid if len(ov_ghid) >= len(ov_ids) else ov_ids

langs = Counter(winner_map[i][0] for i in winner_ov)
tot = sum(langs.values())
print("Matched ids language distribution:")
for l, c in langs.most_common():
    print(f"  {l:10s} : {c:>6,}  ({c/tot*100:.1f}%)")
print(f"\nPython rate: {langs.get('py',0)/tot*100:.1f}%  (expected ~10-18% agar join real)")

Matched ids language distribution:
  java       :  9,447  (22.5%)
  py         :  8,738  (20.8%)
  go         :  8,383  (20.0%)
  js         :  3,817  (9.1%)
  .cs        :  3,438  (8.2%)
  cpp        :  3,059  (7.3%)
  php        :  2,631  (6.3%)
  rb         :  1,523  (3.6%)
  c          :    977  (2.3%)

Python rate: 20.8%  (expected ~10-18% agar join real)


In [25]:
# Ek matched id pe ref ka oldf vs msg ka oldf compare karo — same sample hai?
ref_oldf_by_id = {}
with RemoteZip(ref_url) as z:
    with z.open(member) as f:
        for line in f:
            s = json.loads(line)
            ids_val = s.get('ids')
            if isinstance(ids_val, list) and len(ids_val) > 0:
                ref_oldf_by_id[ids_val[0]] = s.get('oldf', '')[:200]  # first 200 chars

# msg-train se same id ka oldf nikalo
check_id = next(iter(msg_train_ids & set(ref_oldf_by_id.keys())))

msg_oldf = None
with open(train_file, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        if s['id'] == check_id:
            msg_oldf = s.get('oldf', '')[:200]
            break

print(f"Checking id: {check_id}\n")
print("MSG oldf (first 200):", repr(msg_oldf))
print("\nREF oldf (first 200):", repr(ref_oldf_by_id[check_id]))
print("\n→ Agar dono oldf SAME hain, join 100% content-verified ✅")

Checking id: 131075

MSG oldf (first 200): 'from __future__ import print_function, absolute_import, division  # makes KratosMultiphysics backward compatible with python 2.6 and 2.7\n\n# Importing the Kratos Library\nimport KratosMultiphysics\n\n# ot'

REF oldf (first 200): '<?php\n/**\n * Copyright © Magento, Inc. All rights reserved.\n * See COPYING.txt for license details.\n */\nnamespace Magento\\Catalog\\Model;\n\n/**\n * Catalog image uploader\n */\nclass ImageUploader\n{\n    /*'

→ Agar dono oldf SAME hain, join 100% content-verified ✅


In [ ]:
import json, hashlib
from remotezip import RemoteZip

def h(text):
    return hashlib.md5(text.encode('utf-8', errors='ignore')).hexdigest()

ref_url = "https://zenodo.org/records/6900648/files/Code_Refinement.zip?download=1"
member  = "Code_Refinement/ref-train.jsonl"

# ref-train se: hash(oldf) -> (lang, repo)
oldf_to_meta = {}
print("Building oldf->meta map from ref-train ...")
with RemoteZip(ref_url) as z:
    with z.open(member) as f:
        for i, line in enumerate(f):
            s = json.loads(line)
            oldf = s.get('oldf')
            if oldf:
                oldf_to_meta[h(oldf)] = (s.get('lang'), s.get('repo'))
            if (i+1) % 50000 == 0:
                print(f"  {i+1:,} read, {len(oldf_to_meta):,} unique files")

print(f"\nref-train unique oldf files: {len(oldf_to_meta):,}")

Building oldf->meta map from ref-train ...
  50,000 read, 48,788 unique files


In [4]:
import json, hashlib
from pathlib import Path

def h(text):
    return hashlib.md5(text.encode('utf-8', errors='ignore')).hexdigest()

# Project root dhoondo (cot-code-review folder)
# Notebook 'notebooks/' mein hai, toh parent = project root
CWD = Path.cwd()
print(f"Current dir: {CWD}")

# Do possibilities try karo
candidates = [
    CWD.parent / "data" / "raw" / "Code_Refinement" / "ref-train.jsonl",  # agar notebooks/ ke andar se chal raha
    CWD / "data" / "raw" / "Code_Refinement" / "ref-train.jsonl",          # agar root se chal raha
]

ref_train_local = None
for c in candidates:
    print(f"  Trying: {c}  →  exists? {c.exists()}")
    if c.exists():
        ref_train_local = c
        break

if ref_train_local is None:
    print("\n❌ Path nahi mila. Niche wala manual check chala.")
else:
    print(f"\n✅ Mil gaya: {ref_train_local}")

Current dir: d:\College\NCI\Semester 2\Practicum\Semester 3\code\cot-code-review\notebooks
  Trying: d:\College\NCI\Semester 2\Practicum\Semester 3\code\cot-code-review\data\raw\Code_Refinement\ref-train.jsonl  →  exists? True

✅ Mil gaya: d:\College\NCI\Semester 2\Practicum\Semester 3\code\cot-code-review\data\raw\Code_Refinement\ref-train.jsonl


In [5]:
oldf_to_meta = {}
print("Building oldf->meta map...")
with open(ref_train_local, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        s = json.loads(line)
        oldf = s.get('oldf')
        if oldf:
            oldf_to_meta[h(oldf)] = (s.get('lang'), s.get('repo'))
        if (i+1) % 50000 == 0:
            print(f"  {i+1:,} read, {len(oldf_to_meta):,} unique files")

print(f"\nref-train unique oldf files: {len(oldf_to_meta):,}")

Building oldf->meta map...
  50,000 read, 48,788 unique files
  100,000 read, 95,457 unique files
  150,000 read, 140,374 unique files

ref-train unique oldf files: 140,737


In [6]:
from collections import Counter

# msg-train ka path — same folder structure se
msg_train_local = ref_train_local.parent.parent / "Comment_Generation" / "msg-train.jsonl"
print(f"msg-train: {msg_train_local}  →  exists? {msg_train_local.exists()}\n")

matched, unmatched = 0, 0
py_check = Counter()

with open(msg_train_local, 'r', encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        key = h(s.get('oldf', ''))
        if key in oldf_to_meta:
            matched += 1
            py_check[oldf_to_meta[key][0]] += 1
        else:
            unmatched += 1

total = matched + unmatched
print(f"msg-train rows         : {total:,}")
print(f"MATCHED (content-join) : {matched:,}  ({matched/total*100:.1f}%)")
print(f"unmatched              : {unmatched:,}")
print(f"\nMatched Python rows    : {py_check.get('py', 0):,}")
print("\nLang distribution of matched:")
for l, c in py_check.most_common():
    print(f"  {l:10s}: {c:>6,} ({c/matched*100:.1f}%)")

msg-train: d:\College\NCI\Semester 2\Practicum\Semester 3\code\cot-code-review\data\raw\Comment_Generation\msg-train.jsonl  →  exists? True

msg-train rows         : 117,739
MATCHED (content-join) : 32  (0.0%)
unmatched              : 117,707

Matched Python rows    : 9

Lang distribution of matched:
  java      :     14 (43.8%)
  py        :      9 (28.1%)
  go        :      6 (18.8%)
  .cs       :      2 (6.2%)
  cpp       :      1 (3.1%)


In [8]:
import json
from pathlib import Path

# msg files ka folder (ref_train_local se derive)
CG_DIR = ref_train_local.parent.parent / "Comment_Generation"

valid_file = CG_DIR / "msg-valid.jsonl"
test_file  = CG_DIR / "msg-test.jsonl"
train_file = CG_DIR / "msg-train.jsonl"

print(f"valid: {valid_file.exists()}")
print(f"test : {test_file.exists()}")
print(f"train: {train_file.exists()}")

valid: True
test : True
train: True


In [9]:
# val aur test ke Python repo-sets ka overlap
valid_repos, test_repos = set(), set()

with open(valid_file, encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        if s.get('lang') == 'py':
            valid_repos.add(s.get('proj'))

with open(test_file, encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        if s.get('lang') == 'py':
            test_repos.add(s.get('proj'))

overlap = valid_repos & test_repos
print(f"Valid Python repos : {len(valid_repos)}")
print(f"Test Python repos  : {len(test_repos)}")
print(f"Overlap            : {len(overlap)}")
if overlap:
    print(f"Overlapping repos  : {list(overlap)[:10]}")

print(f"\n→ Overlap ~0  : Microsoft splits repo-disjoint (Option 1 strong)")
print(f"→ Overlap bada: splits repo-aware nahi the")

Valid Python repos : 29
Test Python repos  : 28
Overlap            : 28
Overlapping repos  : ['holoviz-holoviews', 'dagster-io-dagster', 'h2oai-datatable', 'spesmilo-electrum', 'kantord-LibreLingo', 'localstack-localstack', 'qutebrowser-qutebrowser', 'PyCQA-pylint', 'databricks-koalas', 'open-mmlab-mmdetection']

→ Overlap ~0  : Microsoft splits repo-disjoint (Option 1 strong)
→ Overlap bada: splits repo-aware nahi the


In [10]:
import json

# msg-train ke pehle 3 samples ka PURA patch field dekho (truncate nahi)
print("Train samples ke patch field (raw):\n")
with open(train_file, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        s = json.loads(line)
        patch = s.get('patch', '')
        print(f"{'='*70}")
        print(f"SAMPLE {i}  (id={s.get('id')})")
        print(f"{'='*70}")
        print(f"patch (first 400 chars):")
        print(repr(patch[:400]))   # repr taaki \n, escape chars dikhein
        print()

Train samples ke patch field (raw):

SAMPLE 0  (id=90017)
patch (first 400 chars):
'@@ -595,8 +595,10 @@ namespace Kratos\n             array_1d<double, 3> b = ZeroVector(3);\n             b[0] = 1.0;\n \n-            const array_1d<double, 3>  c = MathUtils<double>::CrossProduct(a, b);\n-            const array_1d<double, 3>  d = MathUtils<double>::UnitCrossProduct(a, b);\n+            array_1d<double, 3>  c, d;\n+\n+            MathUtils<double>::CrossProduct(c, b, a);\n+            Mat'

SAMPLE 1  (id=144914)
patch (first 400 chars):
'@@ -22,8 +22,13 @@\n For internal use only; no backwards-compatibility guarantees.\n """\n \n+from __future__ import absolute_import\n+\n import sys\n import traceback\n+from builtins import next\n+from builtins import object\n+from builtins import zip\n \n import six\n \n'

SAMPLE 2  (id=12959)
patch (first 400 chars):
'@@ -25,13 +25,16 @@ module View\n       def should_render_revenue?\n         revenue = @tile.revenue_to_render\n \n+        # special

In [11]:
# Pehle sample ke SAARE fields ka content type/preview — kahin aur filename to nahi
with open(train_file, encoding='utf-8') as f:
    s = json.loads(f.readline())

print("Train sample ke saare fields:\n")
for k, v in s.items():
    v_str = str(v)
    print(f"  {k:8s} ({type(v).__name__}, len={len(v_str)}): {repr(v_str[:120])}")

Train sample ke saare fields:

  oldf     (str, len=21889): "//    |  /           |\n//    ' /   __| _` | __|  _ \\   __|\n//    . \\  |   (   | |   (   |\\__ `\n//   _|\\_\\_|  \\__,_|\\__|\\"
  patch    (str, len=545): '@@ -595,8 +595,10 @@ namespace Kratos\n             array_1d<double, 3> b = ZeroVector(3);\n             b[0] = 1.0;\n \n-  '
  msg      (str, len=82): 'I assumed that for CrossProduct the values were inverted as well... Is that right?'
  id       (int, len=5): '90017'
  y        (int, len=1): '1'


In [12]:
import json, hashlib
from remotezip import RemoteZip

def h(text):
    return hashlib.md5(text.encode('utf-8', errors='ignore')).hexdigest()

dqe_url = "https://zenodo.org/records/6900648/files/Diff_Quality_Estimation.zip?download=1"
member  = "Diff_Quality_Estimation/cls-train-chunk-1.jsonl"

# cls chunk-1 se: hash(oldf) -> lang   (content-based, id ignore)
cls_oldf_to_lang = {}
print("Streaming cls-train-chunk-1 (oldf content)...")
with RemoteZip(dqe_url) as z:
    with z.open(member) as f:
        for i, line in enumerate(f):
            s = json.loads(line)
            oldf = s.get('oldf')
            if oldf and 'lang' in s:
                cls_oldf_to_lang[h(oldf)] = s.get('lang')
            if (i+1) % 50000 == 0:
                print(f"  {i+1:,} read, {len(cls_oldf_to_lang):,} unique files")

print(f"\ncls chunk-1 unique oldf files with lang: {len(cls_oldf_to_lang):,}")

Streaming cls-train-chunk-1 (oldf content)...
  50,000 read, 16,438 unique files

cls chunk-1 unique oldf files with lang: 19,924


In [13]:
# msg-train pe content-join (oldf hash)
from collections import Counter

matched, unmatched = 0, 0
lang_dist = Counter()

with open(train_file, encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        key = h(s.get('oldf', ''))
        if key in cls_oldf_to_lang:
            matched += 1
            lang_dist[cls_oldf_to_lang[key]] += 1
        else:
            unmatched += 1

total = matched + unmatched
print(f"msg-train rows         : {total:,}")
print(f"MATCHED (oldf content) : {matched:,}  ({matched/total*100:.1f}%)")
print(f"\nMatched Python rows    : {lang_dist.get('py', 0):,}")
print("\nLang distribution of matched:")
for l, c in lang_dist.most_common():
    print(f"  {l:10s}: {c:>6,} ({c/matched*100:.1f}%)" if matched else "  (none)")

msg-train rows         : 117,739
MATCHED (oldf content) : 11,401  (9.7%)

Matched Python rows    : 3,353

Lang distribution of matched:
  py        :  3,353 (29.4%)
  js        :  1,929 (16.9%)
  go        :  1,790 (15.7%)
  java      :  1,542 (13.5%)
  .cs       :    896 (7.9%)
  cpp       :    833 (7.3%)
  rb        :    482 (4.2%)
  php       :    458 (4.0%)
  c         :    118 (1.0%)


In [14]:
# 3 matched samples pe: msg ka oldf content khud dekho, cls ka lang sahi lagta hai?
checked = 0
with open(train_file, encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        key = h(s.get('oldf',''))
        if key in cls_oldf_to_lang and checked < 3:
            print(f"{'='*60}")
            print(f"cls says lang = {cls_oldf_to_lang[key]}")
            print(f"msg oldf (first 150): {repr(s['oldf'][:150])}")
            print()
            checked += 1
        if checked >= 3:
            break

cls says lang = py
msg oldf (first 150): '"""\nCopyright (c) 2018-2020 Intel Corporation\n\nLicensed under the Apache License, Version 2.0 (the "License");\nyou may not use this file except in com'

cls says lang = .cs
msg oldf (first 150): '\ufeffusing System;\nusing System.Collections.Generic;\nusing System.Diagnostics;\nusing System.IO;\nusing System.Reflection;\nusing System.Windows;\nusing Syste'

cls says lang = java
msg oldf (first 150): '/*\r\n * Licensed to the Apache Software Foundation (ASF) under one or more\r\n * contributor license agreements.  See the NOTICE file distributed with\r\n '



In [15]:
# Pehle try
!pip install guesslang -q

ERROR: Cannot install guesslang==0.9.1, guesslang==0.9.3, guesslang==2.0.0, guesslang==2.0.1, guesslang==2.0.3, guesslang==2.2.0 and guesslang==2.2.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [16]:
!pip install pygments -q

In [17]:
from pygments.lexers import guess_lexer
from pygments.util import ClassNotFound
from collections import Counter, defaultdict

# pygments output → dataset ke lang code ka map
# pygments lexer ke 'aliases' use karenge matching ke liye
PYG_TO_CODE = {
    'python': 'py', 'python2': 'py', 'numpy': 'py',
    'java': 'java',
    'go': 'go',
    'javascript': 'js', 'js': 'js', 'typescript': 'js',
    'csharp': '.cs', 'c#': '.cs',
    'cpp': 'cpp', 'c++': 'cpp',
    'ruby': 'rb', 'rb': 'rb',
    'php': 'php',
    'c': 'c',
}

def detect_lang(code):
    """pygments se language detect karo, dataset code mein map karo."""
    try:
        lexer = guess_lexer(code)
        # lexer ke saare aliases check karo
        for alias in lexer.aliases:
            if alias.lower() in PYG_TO_CODE:
                return PYG_TO_CODE[alias.lower()]
        # name bhi try karo
        name = lexer.name.lower()
        return PYG_TO_CODE.get(name, 'OTHER')
    except ClassNotFound:
        return 'UNKNOWN'
    except Exception:
        return 'ERROR'

# 11,401 ground-truth labeled samples pe test
correct = 0
total = 0
confusion = defaultdict(Counter)   # true_lang -> {predicted: count}

print("Running pygments on ground-truth labeled rows...")
with open(train_file, encoding='utf-8') as f:
    for line in f:
        s = json.loads(line)
        key = h(s.get('oldf', ''))
        if key in cls_oldf_to_lang:
            true_lang = cls_oldf_to_lang[key]
            pred_lang = detect_lang(s.get('oldf', ''))
            confusion[true_lang][pred_lang] += 1
            if pred_lang == true_lang:
                correct += 1
            total += 1
            if total % 2000 == 0:
                print(f"  {total:,} done, running acc: {correct/total*100:.1f}%")

print(f"\n{'='*50}")
print(f"OVERALL ACCURACY: {correct:,}/{total:,} = {correct/total*100:.1f}%")
print(f"{'='*50}")

Running pygments on ground-truth labeled rows...
  2,000 done, running acc: 26.5%
  4,000 done, running acc: 26.7%
  6,000 done, running acc: 26.2%
  8,000 done, running acc: 26.3%
  10,000 done, running acc: 26.3%

OVERALL ACCURACY: 3,001/11,401 = 26.3%
